# ML Project 1: End-to-End Classical ML
## Interview-Oriented Practical Project

This notebook shows how the concepts from the previous ML modules fit together in one real project.

### Topics covered
- Problem definition
- EDA and data quality
- Missing values
- Duplicates
- Class imbalance
- Train/test split
- Preprocessing
- Pipeline
- Baseline
- Logistic Regression
- Decision Tree
- Random Forest
- Cross-validation
- Hyperparameter tuning
- Final evaluation
- Confusion matrix
- Error analysis
- Permutation importance
- Model saving/loading
- Inference
- Interview explanation


In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.inspection import permutation_importance

import joblib

RANDOM_STATE = 42


# 2. PROBLEM DEFINITION

Example problem:

> Predict whether a bank customer will subscribe to a term deposit.

This is a **binary classification** problem.

### Interview approach

Before writing code, identify:

1. What are we predicting?
2. What is the target column?
3. What type of ML problem is this?
4. Which metric matters?
5. What is the cost of false positives and false negatives?


In [2]:
# ============================================================
# 3. LOAD DATA
# ============================================================

# Bank Marketing dataset from OpenML.
# If internet access is unavailable, replace this with your CSV.

data = fetch_openml(
    "bank-marketing",
    version=1,
    as_frame=True
)

df = data.frame.copy()

# OpenML version 1 calls the target column `Class`, not `y`.
# Normalize it so the rest of the notebook uses one consistent name.
df = df.rename(columns={data.target.name: "y"})

print("Shape:", df.shape)
df.head()


Shape: (45211, 17)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,Class
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,1
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,1
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,1
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,1
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,1


In [3]:
# ============================================================
# 4. BASIC DATA INSPECTION
# ============================================================

print("Shape:", df.shape)
print()

print("Columns:")
print(df.columns.tolist())
print()

print("Data types:")
print(df.dtypes)
print()

print("First rows:")
display(df.head())


Shape: (45211, 17)

Columns:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'Class']

Data types:
V1          int64
V2       category
V3       category
V4       category
V5       category
V6          int64
V7       category
V8       category
V9       category
V10         int64
V11      category
V12         int64
V13         int64
V14         int64
V15         int64
V16      category
Class    category
dtype: object

First rows:


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,Class
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,1
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,1
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,1
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,1
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,1


In [4]:
# ============================================================
# 5. SUMMARY STATISTICS
# ============================================================

display(df.describe(include="all").T)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
V1,45211.0,NaN,NaN,NaN,40.93621,10.618762,18.0,33.0,39.0,48.0,95.0
V2,45211,12,blue-collar,9732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
V3,45211,3,married,27214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
V4,45211,4,secondary,23202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
V5,45211,2,no,44396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
V6,45211.0,NaN,NaN,NaN,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
V7,45211,2,yes,25130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
V8,45211,2,no,37967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
V9,45211,3,cellular,29285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
V10,45211.0,NaN,NaN,NaN,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0


In [5]:
# ============================================================
# 6. MISSING VALUES
# ============================================================

missing = df.isnull().sum().sort_values(ascending=False)

print(missing[missing > 0])


Series([], dtype: int64)


In [6]:
# ============================================================
# 7. DUPLICATES
# ============================================================

print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [7]:
# ============================================================
# 8. TARGET DISTRIBUTION
# ============================================================

print(df["y"].value_counts())
print()

print("Class proportions:")
print(df["y"].value_counts(normalize=True))


KeyError: 'y'

# 9. CLASS IMBALANCE

Suppose:

```text
No  = 90%
Yes = 10%
```

A model predicting everything as `No` gets 90% accuracy but detects no positive cases.

Therefore, for imbalanced classification, do not blindly optimize accuracy.

Useful metrics:
- Precision
- Recall
- F1
- ROC-AUC
- PR-AUC when appropriate


In [ ]:
# ============================================================
# 10. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop("y", axis=1)

# This OpenML copy encodes the negative and positive classes as `1` and `2`;
# other copies use `no` and `yes`. Encode the positive label robustly.
target = df["y"].astype(str)
positive_label = "yes" if "yes" in set(target) else sorted(target.unique())[-1]
y = (target == positive_label).astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# ============================================================
# 11. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print()
print("Train positive ratio:", y_train.mean())
print("Test positive ratio :", y_test.mean())


# 12. WHY SPLIT BEFORE PREPROCESSING?

Never fit preprocessing on the complete dataset before splitting.

Wrong:

```python
scaler.fit(X)
train_test_split(...)
```

The scaler has already seen test information.

Correct:

```text
Split
  ↓
Fit preprocessing on train
  ↓
Transform train
  ↓
Transform test
```

Using `Pipeline` helps prevent this leakage during cross-validation.


In [ ]:
# ============================================================
# 13. IDENTIFY NUMERICAL / CATEGORICAL FEATURES
# ============================================================

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print()

print("Categorical features:")
print(categorical_features)


# 14. PREPROCESSING

Numerical:

```text
Missing values
      ↓
Median
      ↓
StandardScaler
```

Categorical:

```text
Missing values
      ↓
Most frequent
      ↓
OneHotEncoder
```

### Why OneHotEncoder?

Models cannot directly use values such as:

```text
job = admin
job = student
job = technician
```

They need numerical representation.


In [ ]:
# ============================================================
# 15. PREPROCESSING PIPELINES
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print(preprocessor)


# 16. BASELINE MODEL

Always establish a simple baseline.

`DummyClassifier(strategy="most_frequent")` predicts the majority class.

If our ML model cannot meaningfully beat the baseline, investigate:
- Data quality
- Features
- Target definition
- Metric
- Model choice


In [ ]:
# ============================================================
# 17. BASELINE
# ============================================================

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))
print("Baseline F1:", f1_score(y_test, baseline_pred))


# 18. LOGISTIC REGRESSION

Good first classification model.

Advantages:
- Simple
- Fast
- Interpretable
- Strong baseline
- Probability output

For this dataset we put preprocessing and the model inside one Pipeline.


In [ ]:
# ============================================================
# 19. LOGISTIC REGRESSION
# ============================================================

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_pipeline.fit(X_train, y_train)

pred = logistic_pipeline.predict(X_test)
prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1       :", f1_score(y_test, pred))
print("ROC-AUC  :", roc_auc_score(y_test, prob))


# 20. DECISION TREE

Decision Tree can learn nonlinear relationships.

Important parameters:

- `max_depth` → maximum tree depth
- `min_samples_split` → minimum samples needed to split a node
- `min_samples_leaf` → minimum samples allowed in a leaf
- `criterion` → split quality measure

Main problem:

> An unrestricted tree can overfit.


In [ ]:
# ============================================================
# 21. DECISION TREE
# ============================================================

tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=10,
        random_state=RANDOM_STATE
    ))
])

tree_pipeline.fit(X_train, y_train)

pred = tree_pipeline.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1       :", f1_score(y_test, pred))


# 22. RANDOM FOREST

Random Forest = many decision trees + aggregation.

```text
Training data
     ↓
Many trees
     ↓
Different samples/features
     ↓
Combine predictions
     ↓
Final prediction
```

Important parameters:

- `n_estimators` → number of trees
- `max_depth` → tree complexity
- `min_samples_split`
- `min_samples_leaf`
- `max_features`
- `class_weight`


In [ ]:
# ============================================================
# 23. RANDOM FOREST
# ============================================================

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

pred = rf_pipeline.predict(X_test)
prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1       :", f1_score(y_test, pred))
print("ROC-AUC  :", roc_auc_score(y_test, prob))


# 24. COMPARE MODELS

Do not compare only training accuracy.

Use validation/cross-validation and the metric appropriate for the business problem.


In [ ]:
# ============================================================
# 25. SIMPLE MODEL COMPARISON
# ============================================================

models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": tree_pipeline,
    "Random Forest": rf_pipeline
}

for name, model in models.items():

    pred = model.predict(X_test)

    print(
        f"{name:20s} "
        f"F1 = {f1_score(y_test, pred):.4f}"
    )


# 26. CROSS-VALIDATION

One train/validation split may be lucky or unlucky.

Use StratifiedKFold for classification.

```text
Fold 1 → validation
Fold 2 → validation
Fold 3 → validation
Fold 4 → validation
Fold 5 → validation

Final score = average
```


In [ ]:
# ============================================================
# 27. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

print("Fold scores:", scores)
print("Mean F1:", scores.mean())
print("Std:", scores.std())


# 28. HYPERPARAMETER TUNING

Common approaches:

- GridSearchCV → tests all specified combinations
- RandomizedSearchCV → tests a random subset
- Bayesian optimization → chooses promising configurations intelligently

For interview-level sklearn work, understand GridSearchCV and RandomizedSearchCV well.


In [ ]:
# ============================================================
# 29. RANDOMIZED SEARCH
# ============================================================

param_distributions = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [5, 8, 12, 16, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", "log2", None]
}

search_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)

print()
print("Best CV F1:")
print(search.best_score_)


# 30. FINAL MODEL

Important principle:

The test set should not influence model selection or hyperparameter tuning.

```text
Train data
    ↓
Cross-validation
    ↓
Hyperparameter tuning
    ↓
Best model
    ↓
Untouched test set
    ↓
Final evaluation
```


In [ ]:
# ============================================================
# 31. FINAL TEST EVALUATION
# ============================================================

final_model = search.best_estimator_

final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, final_pred))
print("Precision:", precision_score(y_test, final_pred))
print("Recall   :", recall_score(y_test, final_pred))
print("F1       :", f1_score(y_test, final_pred))
print("ROC-AUC  :", roc_auc_score(y_test, final_prob))

print()
print("Classification Report:")
print(classification_report(y_test, final_pred))


# 32. CONFUSION MATRIX

```text
                 Actual
              0          1

Predicted 0   TN         FN
Predicted 1   FP         TP
```

TN → correctly predicted negative  
FP → predicted positive but actually negative  
FN → predicted negative but actually positive  
TP → correctly predicted positive


In [ ]:
# ============================================================
# 33. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, final_pred)

print(cm)


# 34. ERROR ANALYSIS

Metrics tell us **how much** the model gets wrong.

Error analysis asks:

> Which examples are wrong and why?

Investigate:
- False positives
- False negatives
- Specific customer groups
- Missing values
- Outliers
- Suspicious patterns
- Potential leakage


In [ ]:
# ============================================================
# 35. INSPECT WRONG PREDICTIONS
# ============================================================

errors = X_test.copy()

errors["actual"] = y_test
errors["predicted"] = final_pred
errors["correct"] = (
    errors["actual"] == errors["predicted"]
)

wrong_predictions = errors[
    errors["correct"] == False
]

print("Number of errors:", len(wrong_predictions))

display(wrong_predictions.head())


# 36. PERMUTATION IMPORTANCE

Question:

> Which original input features are important for the final model's performance?

We shuffle one original feature at a time and measure the performance decrease.


In [ ]:
# ============================================================
# 37. PERMUTATION IMPORTANCE
# ============================================================

result = permutation_importance(
    final_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="f1",
    n_jobs=-1
)

perm = pd.Series(
    result.importances_mean,
    index=X_test.columns
).sort_values(ascending=False)

print(perm.head(15))


# 38. RANDOM FOREST FEATURE IMPORTANCE

Because preprocessing creates one-hot encoded columns, the final model sees transformed features.

`get_feature_names_out()` lets us recover those transformed names.


In [ ]:
# ============================================================
# 39. TRANSFORMED FEATURE IMPORTANCE
# ============================================================

rf_model = final_model.named_steps["model"]
fitted_preprocessor = final_model.named_steps["preprocessor"]

feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

tree_importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print(tree_importance.head(20))


# 40. SAVE THE COMPLETE PIPELINE

Save the complete pipeline rather than only the model.

Why?

Production inference needs:

```text
Raw input
    ↓
Imputation
    ↓
Encoding
    ↓
Scaling
    ↓
Model
    ↓
Prediction
```

Saving the pipeline keeps all preprocessing steps together.


In [ ]:
# ============================================================
# 41. SAVE MODEL
# ============================================================

joblib.dump(
    final_model,
    "bank_marketing_model.joblib"
)

print("Model saved.")


In [ ]:
# ============================================================
# 42. LOAD MODEL
# ============================================================

loaded_model = joblib.load(
    "bank_marketing_model.joblib"
)

print("Model loaded.")


# 43. INFERENCE ON NEW DATA

In production, raw input should have the same feature schema used during training.

The saved Pipeline automatically performs preprocessing before prediction.


In [ ]:
# ============================================================
# 44. PREDICT NEW DATA
# ============================================================

new_customer = X_test.iloc[[0]].copy()

prediction = loaded_model.predict(new_customer)
probability = loaded_model.predict_proba(new_customer)[:, 1]

print("Prediction:", prediction[0])
print("Probability:", probability[0])


# 45. COMPLETE ML WORKFLOW

```text
Business Problem
       ↓
Define Target
       ↓
Collect Data
       ↓
EDA
       ↓
Data Quality
 ├── Missing values
 ├── Duplicates
 ├── Class imbalance
 └── Outliers
       ↓
Train/Test Split
       ↓
Preprocessing
       ↓
Pipeline
       ↓
Baseline
       ↓
Simple Model
       ↓
Multiple Models
       ↓
Cross Validation
       ↓
Hyperparameter Tuning
       ↓
Final Test Evaluation
       ↓
Error Analysis
       ↓
Explainability
       ↓
Save Model
       ↓
Deploy
       ↓
Monitor
```


# 46. INTERVIEW — "WALK ME THROUGH YOUR PROJECT"

Use this structure:

### 1. Problem
"We needed to predict whether a customer would subscribe to a term deposit, so this was a binary classification problem."

### 2. Data
"I checked distributions, missing values, duplicates and class imbalance."

### 3. Split
"I used a stratified train/test split."

### 4. Preprocessing
"I used median imputation and scaling for numerical features and imputation plus one-hot encoding for categorical features."

### 5. Models
"I established a baseline and compared Logistic Regression, Decision Tree and Random Forest."

### 6. Validation
"I used Stratified K-Fold cross-validation."

### 7. Tuning
"I tuned Random Forest hyperparameters using RandomizedSearchCV."

### 8. Evaluation
"I kept the test set untouched until final evaluation and measured precision, recall, F1 and ROC-AUC."

### 9. Error analysis
"I inspected incorrect predictions and feature importance."

### 10. Deployment
"I saved the complete preprocessing and model pipeline so production receives the same transformations as training."


# 47. INTERVIEW CHEAT SHEET

### Why train/test split?
To estimate performance on unseen data.

### Why stratify?
To preserve class distribution.

### Why Pipeline?
To combine preprocessing and model and reduce preprocessing leakage.

### Why ColumnTransformer?
To apply different transformations to numerical and categorical features.

### Why baseline?
To determine whether the ML model adds value over a simple strategy.

### Why cross-validation?
To get a more reliable performance estimate across different splits.

### GridSearch vs RandomizedSearch?
GridSearch tests every specified combination.

RandomizedSearch tests a selected number of sampled combinations and is often more efficient for large search spaces.

### Why keep the test set untouched?
To obtain a final unbiased estimate after model decisions are finished.

### Why F1?
It balances precision and recall and is useful when both matter.

### What is error analysis?
Studying incorrect predictions to discover failure patterns.

### Why save the Pipeline?
Production needs the exact preprocessing and model together.

### What is data leakage?
Information unavailable at prediction time accidentally enters training or evaluation.

### Main goal of ML?
Generalization to unseen real-world data.


# 48. WHAT YOU SHOULD KNOW FOR THE INTERVIEW

### Deep understanding
- End-to-end ML workflow
- Train/validation/test
- Data leakage
- Metrics
- Cross-validation
- Hyperparameter tuning
- Error analysis
- Explainability

### Must be able to code
- `train_test_split`
- `Pipeline`
- `ColumnTransformer`
- `SimpleImputer`
- `StandardScaler`
- `OneHotEncoder`
- Logistic Regression
- Decision Tree
- Random Forest
- `cross_val_score`
- `RandomizedSearchCV`
- Classification metrics
- `permutation_importance`
- `joblib`

### Do not overcomplicate

You do not need to implement every sklearn algorithm from scratch.

For interviews focus on:

**intuition + internal working + assumptions + trade-offs + implementation + debugging + real-world usage.**
